In [48]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from scipy.stats import randint

In [4]:
# data
data = pd.read_csv('../datasets/iris.csv')
data.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [12]:
data['species_index'] = data['species'].map(
    {
        'setosa': 1,
        'versicolor': 2,
        'virginica': 3,
    }
)

In [49]:
# data split
X = data[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y = data['species_index']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# search space
param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [5, 10, 15, None],
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 10)
}

# cv
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# cv search (USES TRAINING DATA ONLY)
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, oob_score=True), # oob is just for comparing and it does not affect our model
    param_distributions=param_dist,
    n_iter=50,           # try 50 random combinations
    cv=cv,               # 5-fold CV on TRAINING data
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# finding best params (hyperparameter tuning) and train final model
random_search.fit(X_train, y_train)

# model (best model)
model = random_search.best_estimator_

# predict
y_pred = model.predict(X_test)

print(f"Best params: {random_search.best_estimator_}")
print(f"Best CV score (on training): {random_search.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: RandomForestClassifier(max_depth=15, min_samples_leaf=4, min_samples_split=14,
                       n_estimators=64, oob_score=True, random_state=42)
Best CV score (on training): 0.9667


In [50]:
target_names=['setosa', 'versicolor', 'virginica']

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# accuracy
test_accuracy = accuracy_score(y_test, y_pred)
train_accuracy = accuracy_score(y_train, y_pred_train)
gap = train_accuracy - test_accuracy
print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"OOB Score (compare with test accuracy): {model.oob_score_:.2%}")
print(f"Overfitting Gap: {gap:.4f}")


if gap < 0.01:
    print("✅ NO OVERFITTING (gap < 1%)")
    status = "Balanced"
elif gap < 0.02:
    print("✅ MINIMAL OVERFITTING (1-2%)")
    status = "Slight Overfitting"
elif gap < 0.03:
    print("⚠️  MILD OVERFITTING (2-3%)")
    status = "Mild Overfitting"
elif gap < 0.05:
    print("⚠️  MODERATE OVERFITTING (3-5%)")
    status = "Moderate Overfitting"
elif gap < 0.10:
    print("🔴 SIGNIFICANT OVERFITTING (5-10%)")
    status = "Significant Overfitting"
else:
    print("🔴 SEVERE OVERFITTING (>10%)")
    status = "Severe Overfitting"

if train_accuracy < 0.70 and test_accuracy < 0.70:
    print("📉 UNDERFITTING DETECTED: Model is too simple")
elif train_accuracy < 0.75:
    print("⚠️  Possible underfitting (training accuracy < 75%)")
else:
    print("✅ No underfitting detected")

# detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
correct_predictions = np.trace(cm)
total_predictions = np.sum(cm)
accuracy = correct_predictions / total_predictions

print("\nConfusion Matrix:")
print(cm)


print(f"Correct predictions: {correct_predictions}")
print(f"Total predictions: {total_predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy*100:.2f}%")

for i in range(cm.shape[0]):
    class_correct = cm[i, i]
    class_total = np.sum(cm[i, :])
    accuracy = class_correct / class_total
    print(f"{target_names[i]} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")



MODEL EVALUATION
Training Accuracy: 98.33%
Test Accuracy: 96.67%
OOB Score (compare with test accuracy): 95.00%
Overfitting Gap: 0.0167
✅ MINIMAL OVERFITTING (1-2%)
✅ No underfitting detected

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30


Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
Correct predictions: 29
Total predictions: 30
Accuracy: 0.9667
Accuracy: 96.67%
setosa Accuracy: 1.0000 (100.00%)
versicolor Accuracy: 0.9000 (90.00%)
virginica Accuracy: 1.0000 (100.00%)


In [51]:
print("\n" + "="*60)
print("FEATURE IMPORTANCE")
print("="*60)

importance = model.feature_importances_
for name, imp in zip(X.columns, importance):
    print(f"{name}: {imp:.4f}")

print(model.feature_importances_)


FEATURE IMPORTANCE
sepal_length: 0.1244
sepal_width: 0.0052
petal_length: 0.4235
petal_width: 0.4470
[0.12435645 0.00516808 0.42351936 0.44695611]
